# 19 · A warm chocolate bar bends 🍫🔥

In notebook 13 the chocolate bar bent under **gravity**. Now we bend it with **heat**.
Warm its peaks and cool its base: the warm material wants to expand more than the
cold material, and since the two are bonded into one bar, that mismatch makes the
whole thing **curl** — no external force at all. This is our first genuinely
**coupled** problem: a temperature field $T$ and a displacement field $\mathbf u$,
solved one after the other, the first **feeding** the second. It is exactly how a
bimetallic thermostat, a warped circuit board, or a cooling weld deforms.

In [ ]:
# --- Google Colab: install NGSolve on first run (a no-op anywhere else) -------
# NGSolve ships its PyPI wheels as pre-releases, so the `--pre` flag is essential.
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--pre",
                    "ngsolve", "anywidget"], check=True)

In [ ]:
from netgen.occ import *
from ngsolve import *
from ngsolve.webgui import Draw

## 1. The bar — and which surfaces we heat

The same chocolate bar as notebook 13. We clamp the `x=0` end (it cannot move), and we
label two sets of faces for the **thermal** problem: the upper faces (the peaks) are
kept **warm**, the flat **base** is kept **cool**. Everything in between is left
**insulated** — a natural (no-flux) boundary that needs no equation.

In [ ]:
def chocolate_bar():
    n_peaks, b, depth, base_h, ph, r_fil = 3, 2.0, 3.0, 0.4, 2.6, 0.2
    W = n_peaks*b
    def trap(a, b, c, d):
        return Face(Wire([Segment(a, b), Segment(b, c), Segment(c, d), Segment(d, a)]))
    def peak(cx, cy, z0, bx, by, h):
        rx = Prism(trap(Pnt(cx-bx/2, cy-by/2, z0), Pnt(cx+bx/2, cy-by/2, z0),
                        Pnt(cx+0.1*bx, cy-by/2, z0+h), Pnt(cx-0.1*bx, cy-by/2, z0+h)), Vec(0, by, 0))
        ry = Prism(trap(Pnt(cx-bx/2, cy-by/2, z0), Pnt(cx-bx/2, cy+by/2, z0),
                        Pnt(cx-bx/2, cy+0.1*by, z0+h), Pnt(cx-bx/2, cy-0.1*by, z0+h)), Vec(bx, 0, 0))
        return rx * ry
    solid = Box(Pnt(0, -0.3, 0), Pnt(W, depth + 0.3, base_h))
    for i in range(n_peaks):
        solid = solid + peak((i + 0.5)*b, depth/2, base_h, b, depth + 0.6, ph)
    along_y = lambda e: abs(e.vertices[0].p[1]-e.vertices[1].p[1]) > abs(e.vertices[0].p[0]-e.vertices[1].p[0])
    valleys = [e for e in solid.edges
               if abs(e.center[2]-base_h) < 0.05 and along_y(e) and 0.1 < e.center[0] < W-0.1]
    solid = solid.MakeFillet(valleys, r_fil)
    return solid * Box(Pnt(-1, 0, -1), Pnt(W + 1, depth, 10*ph)), W, depth, base_h

bar, L, depth, base_h = chocolate_bar()
bar.faces.Min(X).name = "clamp"
for f in bar.faces:
    if   f.center[2] > base_h*0.7: f.name = "warm"      # the peaks
    elif f.center[2] < 0.05:       f.name = "cool"      # the flat base
mesh = Mesh(OCCGeometry(bar).GenerateMesh(maxh=1.0)); mesh.Curve(2)
Draw(mesh)

## 2. Step one — the temperature field

A plain **steady heat** problem, $-\nabla\!\cdot(\nabla T)=0$, with the warm peaks
and cool base imposed as Dirichlet data. We set each boundary value separately so
the data is clean (no spurious over/undershoot), then solve the SPD system with
`sparsecholesky`. This $T$ is the **input** to the mechanics.

In [ ]:
fesT = H1(mesh, order=2, dirichlet="warm|cool")
T, s = fesT.TnT()
gfT = GridFunction(fesT)
gfT.Set(mesh.BoundaryCF({"warm": 30.0, "cool": 0.0}),   # both values in one projection,
        definedon=mesh.Boundaries("warm|cool"))         # restricted so side faces stay free

aT = BilinearForm(grad(T)*grad(s)*dx).Assemble()
rT = (-aT.mat*gfT.vec).Evaluate()                       # residual keeps the Dirichlet data
gfT.vec.data += aT.mat.Inverse(fesT.FreeDofs(), inverse="sparsecholesky")*rT
gfP1 = GridFunction(H1(mesh, order=1)); gfP1.Set(gfT)   # nodal values for an honest min/max
print(f"temperature stays within [{min(gfP1.vec):.1f}, {max(gfP1.vec):.1f}] °C "
      f"(imposed: 0 at the base, 30 at the peaks)")
Draw(gfT, mesh, "temperature")

## 3. Step two — the thermal stress that bends the bar

Hooke's law from notebook 13 gains one term. A temperature rise $T-T_0$ makes the
material want to **expand** by $\alpha(T-T_0)$ in every direction; the stress only
responds to the part of the strain *beyond* that free expansion:
$$\sigma = 2\mu\,\varepsilon(\mathbf u) + \lambda\,\mathrm{tr}\,\varepsilon\,I
           \;-\;(3\lambda+2\mu)\,\alpha\,(T-T_0)\,I .$$
Putting this into $\int\sigma:\varepsilon(\mathbf v)=0$ and moving the known thermal
part to the right gives an ordinary elasticity solve with a **thermal body load**
$\beta\,(T-T_0)\,\nabla\!\cdot\mathbf v$, where $\beta=(3\lambda+2\mu)\alpha$. Same
SPD operator as before — only the right-hand side changed.

In [ ]:
E, nu, alpha, T0 = 1.0e4, 0.3, 2.0e-3, 0.0
mu  = E/(2*(1+nu))
lam = E*nu/((1+nu)*(1-2*nu))
beta = (3*lam + 2*mu)*alpha                             # thermal stress coefficient

def strain(u): return Sym(Grad(u))
def stress(u): return 2*mu*strain(u) + lam*Trace(strain(u))*Id(3)

fes = VectorH1(mesh, order=2, dirichlet="clamp")
u, v = fes.TnT()
a = BilinearForm(InnerProduct(stress(u), strain(v))*dx).Assemble()
f = LinearForm(beta*(gfT - T0)*div(v)*dx).Assemble()    # the heat enters here
gfu = GridFunction(fes)
gfu.vec.data = a.mat.Inverse(fes.FreeDofs(), inverse="sparsecholesky")*f.vec

tip = gfu(mesh(L-0.05, depth/2, base_h))
print(f"free-end displacement (x,y,z) = ({tip[0]:+.3f}, {tip[1]:+.3f}, {tip[2]:+.3f})")
print(f"the warm bar curls {abs(tip[2]):.2f} units at its free end — with no applied force")

## 4. See it curl

`Draw(..., deformation=gfu)` warps the bar by the displacement, coloured by the
temperature that caused it. The peaks (warm, expanding) push the bar into a gentle
arch — pure thermo-mechanics, no gravity, no hand.

In [ ]:
Draw(gfT, mesh, "temperature", deformation=gfu)

**Next:** one last coupling, and the sweetest — we let the chocolate **melt**, a
moving front where temperature and phase chase each other.

In [ ]:
# Navigation between units — shown only in a live notebook (Colab / JupyterLite /
# local Jupyter), never in the rendered website (which has its own prev/next nav).
import os, sys
if not os.environ.get("WEBGUI_SCENE_DIR"):          # not the static site build
    _prev = ("18-buoyant-convection", "18 · A glimpse of more: buoyant convection ☕🔥")
    _next = None
    def _u(_nb):
        if "google.colab" in sys.modules:
            return "https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/" + _nb + ".ipynb"
        return _nb + ".ipynb"                       # JupyterLite & local: relative .ipynb link
    _parts  = ["⬅️ **Previous:** [%s](%s)" % (_prev[1], _u(_prev[0]))] if _prev else []
    _parts += ["➡️ **Next:** [%s](%s)" % (_next[1], _u(_next[0]))] if _next else []
    from IPython.display import display, Markdown
    display(Markdown(" · ".join(_parts)))